In [283]:
import numpy as np
import einops as eo
import matplotlib.pyplot as plt

# laod some basic images to play with
data = np.load("./resources/test_images.npy", allow_pickle=False) # [6, 96, 96, 3]

# basic operations from einops
eo.rearrange
eo.reduce
eo.repeat

# TRANSPOSING
# nump -> data[0].transpose(1, 0, 2)
out = eo.rearrange(data[0], "h w c -> w h c")

# CONCATENATING / FLATTENING: 
# numpy -> data.reshape(-1, 96, 3) 
# numpy -> data.transpose(0, 2, 1, 3).reshape(-1, 96, 3).transpose(1, 0, 2) xD
out = eo.rearrange(data, "b h w c -> (b h) w c")
out = eo.rearrange(data, "b h w c -> h (b w) c")

# FLATTENING MULTIPLE
# numpy -> data.reshape(-1, 3)
out = eo.rearrange(data, "b h w c -> (b h w) c") #all pixels in a line

# DECOMPOSITION
out = eo.rearrange(data, "(b1 b2) h w c -> b1 b2 h w c", b1=2, b2=3)
out = eo.rearrange(data, "(b1 b2) h w c -> (b1 h) (b2 w) c", b1=2, b2=3)
out = eo.rearrange(data, "(b1 b2) h w c -> (b2 h) (b1 w) c", b1=2, b2=3)

# WIDTH 2 HEIGHT   
out = eo.rearrange(data, "b h (w1 w2) c -> (h w2) (b w1) c", w2=2) # crazy cursed

# MEAN ALONG AXIS
# numpy -> np.mean(data, axis=0)
out = eo.reduce(data, "b h w c -> h w c", "mean")
out = eo.reduce(data, "b h w c -> h w c", "min")
out = eo.reduce(data, "b h w c -> h w c", "max")
out = eo.reduce(data, "b h w c -> h w c", "sum")
out = eo.reduce(data, "b h w c -> h w c", "prod")

# MEAN POOLING WITH 2X2 KERNEL 
# hint: (h h2) -> h h2 is just reshape(-1, 2) so the small index is "filled first", cutting up the input a lot
# basically this forms a half as long vector of direct pairs of pixels ...
out = eo.reduce(data, "b (h h2) (w w2) c -> h (b w) c", "mean", h2=2, w2=2)
out = eo.reduce(data, "b (h 2) (w 2) c -> h (b w) c", "mean") # synonym!

# ADD SINGLETON DIMENSION
# numpy -> data[None, :, :, :, None, :]
out = eo.rearrange(data, "b h w c -> 1 b h w 1 c" )
out = eo.rearrange(data, "b h w c -> () b h w () c") # synonym

# SQUEEZE (can partially squeeze!)
out = eo.rearrange(out, "1 b h w 1 c -> 1 b h w c") # synonym again; ()

# REPEAT ALONG NEW AXIS
out = eo.repeat(data[0, ...], "h w c -> z h w c", z=10)
out = eo.repeat(data[0, ...], "h w c -> 10 h w c") # synonym 

# REPEAT ALONG EXISTING AXIS
out = eo.repeat(data[0, ...], "h w c -> h (repeat w) c", repeat=3)
out = eo.repeat(data[0, ...], "h w c -> h (3 w) c") # synonym

# ORDER MATTERS (this doesn't repeat the row, but rather each pixel for the whole row)
out = eo.repeat(data[0, ...], "h w c -> h (w 3) c") # synonym

# PACKING / CONCATENATING MULTIPLE ARRAY
A = data[0, :, :, 0:2]
B = data[1, :, :, 2]
out, ps = eo.pack([A, B], "h w *")

# UNPACKING / SPLITTING
out_1, out_2 = eo.unpack(out, ps, "h w *")
out_1, out_2 = eo.unpack(out, [[2], [1]], "h w *") # synonym

# USEFUL FOR AUTOBATCHING / DEALING WITH ARBITRARY STACKING / PACKING -> UNPACKING
# basically collapses all dimensions excep h w c into one... is kinda missusing pack as a variable rearrange
# seems to be veeeeery similar to .reshape(-1, h w c)!

x1 = np.random.rand(96, 96, 3)
x2 = np.random.rand(1, 96, 96, 3)
x3 = np.random.rand(10, 96, 96, 3)
x4 = np.random.rand(3, 10, 96, 96, 3)

def autobatch_einops(x):
    # print(f"input shape: {x.shape}")
    x_batched, ps = eo.pack([x], "* h w c")
    # print(f"x_batched shape: {x_batched.shape}") 
    
    ... # do some operation

    [x_distributed] = eo.unpack(x_batched, ps, "* h w c") # for some reason there's a singleton list around this
    # print(f"output shape: {x_distributed.shape}")
    return x_distributed

def autobatch_numpy(x): 
    # well this is arguably almost more elegant and also veeery self documenting
    # also pure numpy is like 10x faster lol
    # print(f"input shape: {x.shape}")
    *batch_dims, h, w, c = x.shape # batch_dims eats all the surplus dims, but can also be empty
    x_batched = x.reshape(-1, h, w, c)
    # print(f"x_batched shape: {x_batched.shape}") 
    
    ... # do some operation
    
    x_distributed = x_batched.reshape(*batch_dims, h, w, c)
    # print(f"output shape: {x_distributed.shape}")
    return x_distributed

In [7]:
import einops as eo
import numpy as np

A = np.random.rand(10, 100, 8)

print("numpy")
%timeit A.mean(axis=-1)
%timeit A.sum(axis=-1)
%timeit A.max(axis=-1)

print("eo")
%timeit eo.reduce(A, "a b c -> a b", "mean")
%timeit eo.reduce(A, "a b c -> a b", "sum")
%timeit eo.reduce(A, "a b c -> a b", "max")

numpy
21.7 μs ± 77.9 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
15.2 μs ± 144 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
37.1 μs ± 536 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
eo
28 μs ± 314 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
19.4 μs ± 175 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
41.3 μs ± 590 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
